In [4]:
import pandas as pd

# Load only first 500k rows to keep it fast and safe in Colab
df_flights = pd.read_csv('/content/flights.csv', nrows=500000)

print("Shape:", df_flights.shape)
print("\nColumns:", df_flights.columns.tolist())
print("\nFirst 5 rows:")
display(df_flights.head())

print("\nData types and non-null counts:")
df_flights.info()

Shape: (51809, 31)

Columns: ['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER', 'TAIL_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'WHEELS_ON', 'TAXI_IN', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME', 'ARRIVAL_DELAY', 'DIVERTED', 'CANCELLED', 'CANCELLATION_REASON', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']

First 5 rows:


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259.0,-21.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN



Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51809 entries, 0 to 51808
Data columns (total 31 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   YEAR                 51809 non-null  int64  
 1   MONTH                51809 non-null  int64  
 2   DAY                  51809 non-null  int64  
 3   DAY_OF_WEEK          51809 non-null  int64  
 4   AIRLINE              51809 non-null  object 
 5   FLIGHT_NUMBER        51809 non-null  int64  
 6   TAIL_NUMBER          51749 non-null  object 
 7   ORIGIN_AIRPORT       51809 non-null  object 
 8   DESTINATION_AIRPORT  51809 non-null  object 
 9   SCHEDULED_DEPARTURE  51809 non-null  int64  
 10  DEPARTURE_TIME       50675 non-null  float64
 11  DEPARTURE_DELAY      50675 non-null  float64
 12  TAXI_OUT             50651 non-null  float64
 13  WHEELS_OFF           50651 non-null  float64
 14  SCHEDULED_TIME       51809 non-null  int64  
 15  ELA

In [5]:
# Filter for United Airlines (UA) in 2015
ua_2015 = df_flights[(df_flights['AIRLINE'] == 'UA') & (df_flights['YEAR'] == 2015)].copy()

print(f"United Airlines flights in 2015 sample: {len(ua_2015)}")

# Route-level operational KPIs
route_stats = ua_2015.groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']).agg(
    total_flights = ('AIRLINE', 'count'),
    cancellation_rate = ('CANCELLED', 'mean'),        # fraction
    avg_dep_delay = ('DEPARTURE_DELAY', 'mean'),
    avg_arr_delay = ('ARRIVAL_DELAY', 'mean')
).reset_index()

# Convert cancellation rate to percentage and round for readability
route_stats['cancellation_rate'] = (route_stats['cancellation_rate'] * 100).round(2)

# Sort by number of flights to see the busiest UA routes
route_stats = route_stats.sort_values('total_flights', ascending=False)

print("\nTop 10 busiest UA routes (by flight count):")
display(route_stats.head(10))

United Airlines flights in 2015 sample: 4386

Top 10 busiest UA routes (by flight count):


,ORIGIN_AIRPORT,DESTINATION_AIRPORT,total_flights,cancellation_rate,avg_dep_delay,avg_arr_delay
249,LAX,IAH,47,0.0,14.978723,16.468085
413,SFO,ORD,43,0.0,18.023256,14.813953
67,DEN,IAH,42,0.0,22.333333,36.000000
398,SFO,EWR,41,0.0,14.512195,5.243902
197,IAH,LAX,41,0.0,29.365854,18.707317
186,IAH,DEN,40,0.0,27.025000,19.150000
402,SFO,IAH,40,0.0,12.650000,15.075000
258,LAX,ORD,38,0.0,12.736842,5.605263
334,ORD,SFO,38,0.0,19.052632,3.026316
214,IAH,SFO,38,0.0,34.842105,14.315789


In [7]:
# Load airports.csv and keep only the identifiers and readable info
airports = pd.read_csv('airports.csv')
airports = airports[['IATA_CODE', 'AIRPORT', 'CITY', 'STATE']].copy()
airports.columns = ['IATA_CODE', 'AIRPORT_NAME', 'CITY', 'STATE']  # rename for clarity

print("Airports sample:")
display(airports.head())

# Merge airport names for ORIGIN
route_stats_named = route_stats.merge(
    airports, left_on='ORIGIN_AIRPORT', right_on='IATA_CODE', how='left'
).rename(columns={'AIRPORT_NAME': 'ORIGIN_AIRPORT_NAME', 'CITY': 'ORIGIN_CITY', 'STATE': 'ORIGIN_STATE'}).drop(columns='IATA_CODE')

# Merge airport names for DESTINATION
route_stats_named = route_stats_named.merge(
    airports, left_on='DESTINATION_AIRPORT', right_on='IATA_CODE', how='left'
).rename(columns={'AIRPORT_NAME': 'DEST_AIRPORT_NAME', 'CITY': 'DEST_CITY', 'STATE': 'DEST_STATE'}).drop(columns='IATA_CODE')

# Create a human-readable route label
route_stats_named['Route'] = route_stats_named['ORIGIN_CITY'] + ' → ' + route_stats_named['DEST_CITY']

# Re-order columns for display
route_display = route_stats_named[['Route', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT',
                                   'total_flights', 'cancellation_rate', 'avg_dep_delay', 'avg_arr_delay']].sort_values('total_flights', ascending=False)

print("\nTop UA routes with city names:")
display(route_display.head(10))

Airports sample:


,IATA_CODE,AIRPORT_NAME,CITY,STATE
0,ABE,Lehigh Valley International Airport,Allentown,PA
1,ABI,Abilene Regional Airport,Abilene,TX
2,ABQ,Albuquerque International Sunport,Albuquerque,NM
3,ABR,Aberdeen Regional Airport,Aberdeen,SD
4,ABY,Southwest Georgia Regional Airport,Albany,GA



Top UA routes with city names:


,Route,ORIGIN_AIRPORT,DESTINATION_AIRPORT,total_flights,cancellation_rate,avg_dep_delay,avg_arr_delay
0,Los Angeles → Houston,LAX,IAH,47,0.0,14.978723,16.468085
1,San Francisco → Chicago,SFO,ORD,43,0.0,18.023256,14.813953
2,Denver → Houston,DEN,IAH,42,0.0,22.333333,36.000000
4,Houston → Los Angeles,IAH,LAX,41,0.0,29.365854,18.707317
3,San Francisco → Newark,SFO,EWR,41,0.0,14.512195,5.243902
6,San Francisco → Houston,SFO,IAH,40,0.0,12.650000,15.075000
5,Houston → Denver,IAH,DEN,40,0.0,27.025000,19.150000
8,Chicago → San Francisco,ORD,SFO,38,0.0,19.052632,3.026316
10,Houston → Newark,IAH,EWR,38,0.0,25.289474,10.973684
11,Newark → Houston,EWR,IAH,38,0.0,19.868421,34.342105


In [8]:
# From earlier we have ua_2015 (all UA flights in 2015 sample)
# We'll join with route_stats to label each flight with its route rank

# Create a ranked list of top routes (by flight count) from earlier work
top_routes = route_stats.head(10)[['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']].copy()
top_routes['is_top_route'] = True

# Merge to filter flights only on those top 10 routes
ua_top = ua_2015.merge(top_routes, on=['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'], how='inner')

# Group by route and calculate average delay minutes from each cause
delay_cause_cols = ['AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']

cause_stats = ua_top.groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'])[delay_cause_cols].mean().reset_index()

# Merge with airport names for readability
cause_stats_named = cause_stats.merge(
    airports, left_on='ORIGIN_AIRPORT', right_on='IATA_CODE', how='left'
).rename(columns={'CITY': 'Origin City'}).drop(columns=['IATA_CODE', 'AIRPORT_NAME', 'STATE'])

cause_stats_named = cause_stats_named.merge(
    airports, left_on='DESTINATION_AIRPORT', right_on='IATA_CODE', how='left'
).rename(columns={'CITY': 'Dest City'}).drop(columns=['IATA_CODE', 'AIRPORT_NAME', 'STATE'])

# Create Route label
cause_stats_named['Route'] = cause_stats_named['Origin City'] + ' → ' + cause_stats_named['Dest City']

# Reorder columns nicely
cols_order = ['Route', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'] + delay_cause_cols
cause_stats_named = cause_stats_named[cols_order].sort_values('LATE_AIRCRAFT_DELAY', ascending=False)

print("Average delay minutes by cause — Top 10 UA routes:")
display(cause_stats_named.round(1))

Average delay minutes by cause — Top 10 UA routes:


,Route,ORIGIN_AIRPORT,DESTINATION_AIRPORT,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
9,San Francisco → Chicago,SFO,ORD,5.4,0.0,31.9,25.7,0.0
2,Houston → Los Angeles,IAH,LAX,6.9,0.0,23.5,20.5,1.2
5,Los Angeles → Chicago,LAX,ORD,11.2,0.0,19.4,20.4,0.0
7,San Francisco → Newark,SFO,EWR,19.6,0.0,13.9,18.6,0.0
1,Houston → Denver,IAH,DEN,3.7,0.0,32.7,15.3,1.2
3,Houston → San Francisco,IAH,SFO,0.0,0.0,32.9,13.3,0.0
4,Los Angeles → Houston,LAX,IAH,7.6,0.0,17.8,12.8,0.0
6,Chicago → San Francisco,ORD,SFO,0.6,0.0,19.6,6.3,1.7
8,San Francisco → Houston,SFO,IAH,9.8,0.0,17.4,3.9,1.2
0,Denver → Houston,DEN,IAH,18.2,0.0,22.2,3.6,1.0


In [9]:
# Focus on SFO-ORD route
route_sfo_ord = ua_2015[(ua_2015['ORIGIN_AIRPORT'] == 'SFO') & (ua_2015['DESTINATION_AIRPORT'] == 'ORD')].copy()

# Current scheduled block time (SCHEDULED_TIME) and actual elapsed time (ELAPSED_TIME)
print("Current avg scheduled time (min):", route_sfo_ord['SCHEDULED_TIME'].mean().round(0))
print("Current avg actual elapsed time (min):", route_sfo_ord['ELAPSED_TIME'].mean().round(0))
print("Current avg arrival delay (min):", route_sfo_ord['ARRIVAL_DELAY'].mean().round(1))

# Simulate adding 10 minutes to scheduled time
# Under new schedule, a flight that was 10 min late would now be on-time
# We recalculate "new" arrival delay = max(0, old arrival delay - 10)
import numpy as np
route_sfo_ord['NEW_ARRIVAL_DELAY'] = np.maximum(0, route_sfo_ord['ARRIVAL_DELAY'] - 10)

print("\nAfter adding 10 min buffer:")
print("New avg arrival delay (min):", route_sfo_ord['NEW_ARRIVAL_DELAY'].mean().round(1))
print("On-time performance (old):", (route_sfo_ord['ARRIVAL_DELAY'] <= 0).mean().round(2))
print("On-time performance (new):", (route_sfo_ord['NEW_ARRIVAL_DELAY'] <= 0).mean().round(2))

# Impact on total delay minutes (sum of positive delays)
old_total_delay = route_sfo_ord.loc[route_sfo_ord['ARRIVAL_DELAY'] > 0, 'ARRIVAL_DELAY'].sum()
new_total_delay = route_sfo_ord.loc[route_sfo_ord['NEW_ARRIVAL_DELAY'] > 0, 'NEW_ARRIVAL_DELAY'].sum()
reduction_min = old_total_delay - new_total_delay
print(f"\nTotal delay minutes reduced by {reduction_min:.0f} minutes across {len(route_sfo_ord)} flights.")
print(f"Avg improvement per flight: {reduction_min/len(route_sfo_ord):.1f} minutes.")

Current avg scheduled time (min): 252.0
Current avg actual elapsed time (min): 249.0
Current avg arrival delay (min): 14.8

After adding 10 min buffer:
New avg arrival delay (min): 13.8
On-time performance (old): 0.42
On-time performance (new): 0.63

Total delay minutes reduced by 206 minutes across 43 flights.
Avg improvement per flight: 4.8 minutes.


In [10]:
# Create Excel file with multiple sheets
with pd.ExcelWriter('UA_Network_Planning_Analysis.xlsx', engine='openpyxl') as writer:
    route_stats_named.to_excel(writer, sheet_name='Route Performance', index=False)
    cause_stats_named.to_excel(writer, sheet_name='Delay Causes', index=False)

    # Create summary table for SFO-ORD simulation
    sim_summary = pd.DataFrame({
        'Metric': ['Scheduled Block Time (min)', 'Avg Actual Elapsed (min)',
                   'Avg Arrival Delay - Original', 'Avg Arrival Delay - with +10 min buffer',
                   'On-Time Performance - Original', 'On-Time Performance - with +10 min buffer'],
        'Value': [252, 249, 14.8, 13.8, '42%', '63%']
    })
    sim_summary.to_excel(writer, sheet_name='SFO-ORD Buffer Simulation', index=False)

print("Excel file saved. Downloading now...")
from google.colab import files
files.download('UA_Network_Planning_Analysis.xlsx')

Excel file saved. Downloading now...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>